# EDA — TU Delft Server Files

Quick scan of the three metadata files on `\\tudelft.net\staff-umbrella\SLIRotterdam\`.
Goal: understand column names, data types, and how the files can be linked together
so we can resolve `recording_*` filenames to actual server paths for the selected photos.

**Files inspected:**
1. `image_index.csv` — master index with H3 paths for both directional and panorama datasets
2. `internal/import_locations.txt` — tab-delimited panorama location metadata
3. `internal/Panorama_2024_RD/Panorama_2024_RD.shp` — shapefile with panorama locations

**Strategy:** only read the first N rows of each file — never load all 600k+ rows.

In [3]:
import os
import pandas as pd
import geopandas as gpd
import fiona

# Root of the SLI Rotterdam share on the TU Delft network drive
SERVER_ROOT = r"\\tudelft.net\staff-umbrella\SLIRotterdam"

# Paths to the three files we want to inspect
IMAGE_INDEX_PATH    = os.path.join(SERVER_ROOT, "image_index.csv")
IMPORT_LOC_PATH     = os.path.join(SERVER_ROOT, "internal", "import_locations.txt")
SHAPEFILE_PATH      = os.path.join(SERVER_ROOT, "internal", "Panorama_2024_RD", "Panorama_2024_RD.shp")

# Quick existence check so we know early if the network drive is reachable
for label, path in [
    ("image_index.csv",         IMAGE_INDEX_PATH),
    ("import_locations.txt",    IMPORT_LOC_PATH),
    ("Panorama_2024_RD.shp",    SHAPEFILE_PATH),
]:
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else None
    print(f"{'OK' if exists else 'MISSING':8s}  {label:<30s}",
          f"({size_mb:.0f} MB)" if size_mb else "")

OK        image_index.csv                (278 MB)
OK        import_locations.txt           (53 MB)
OK        Panorama_2024_RD.shp           (17 MB)


---
## 1. `image_index.csv` — master path index

This is the largest file (~278 MB). It maps every recording location to its H3 folder
path, which is needed to construct actual file paths on the server.
We only read the first few rows plus dtypes.

In [4]:
# Read just the header + 5 rows — avoids loading 600k rows into memory
idx_peek = pd.read_csv(IMAGE_INDEX_PATH, nrows=5)

print("=== Columns and dtypes ===")
print(idx_peek.dtypes)
print()
print("=== First 5 rows ===")
idx_peek

=== Columns and dtypes ===
image_id             str
x                float64
y                float64
day                int64
month              int64
year               int64
trip_time            str
H3_8                 str
H3_9                 str
H3_10                str
img_front            str
img_back             str
img_left             str
img_right            str
heading_front    float64
heading_right    float64
heading_back     float64
heading_left     float64
dtype: object

=== First 5 rows ===


,image_id,x,y,day,month,year,trip_time,H3_8,H3_9,H3_10,img_front,img_back,img_left,img_right,heading_front,heading_right,heading_back,heading_left
0,1_00001,98448.887,438163.582,25,6,2024,2024-06-25 05:40:31,88196bb733fffff,89196bb7337ffff,8a196bb73347fff,88196bb733fffff/89196bb7337ffff/8a196bb73347ff...,88196bb733fffff/89196bb7337ffff/8a196bb73347ff...,88196bb733fffff/89196bb7337ffff/8a196bb73347ff...,88196bb733fffff/89196bb7337ffff/8a196bb73347ff...,235.433253,325.433253,55.433253,145.433253
1,1_00002,98444.797,438160.764,25,6,2024,2024-06-25 05:40:31,88196bb733fffff,89196bb7337ffff,8a196bb73347fff,88196bb733fffff/89196bb7337ffff/8a196bb73347ff...,88196bb733fffff/89196bb7337ffff/8a196bb73347ff...,88196bb733fffff/89196bb7337ffff/8a196bb73347ff...,88196bb733fffff/89196bb7337ffff/8a196bb73347ff...,235.433253,325.433253,55.433253,145.433253
2,1_00003,98441.052,438157.438,25,6,2024,2024-06-25 05:40:31,88196bb733fffff,89196bb7337ffff,8a196bb7334ffff,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,189.362428,279.362428,9.362428,99.362428
3,1_00004,98440.239,438152.507,25,6,2024,2024-06-25 05:40:31,88196bb733fffff,89196bb7337ffff,8a196bb7334ffff,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,189.362428,279.362428,9.362428,99.362428
4,1_00005,98441.301,438147.582,25,6,2024,2024-06-25 05:40:31,88196bb733fffff,89196bb7337ffff,8a196bb7334ffff,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...,167.094685,257.094685,347.094685,77.094685


In [5]:
# Quick row count via line counting — much faster than loading the full CSV.
# We subtract 1 for the header row.
with open(IMAGE_INDEX_PATH, "r", encoding="utf-8") as f:
    n_rows_index = sum(1 for _ in f) - 1

print(f"Total rows in image_index.csv: {n_rows_index:,}")

Total rows in image_index.csv: 617,579


In [6]:
# Closer look at the key linking columns:
#   image_id  — trip and sequence identifier, used to build the filename
#   trip_time — recording session start, used to match recording_* filenames
#   H3_8/9/10 — H3 hex cell IDs that form the folder structure
#   img_front — example relative path showing the full folder pattern

link_cols = ["image_id", "trip_time", "H3_8", "H3_9", "H3_10", "img_front"]
print("Linking-relevant columns:")
idx_peek[link_cols]

Linking-relevant columns:


,image_id,trip_time,H3_8,H3_9,H3_10,img_front
0,1_00001,2024-06-25 05:40:31,88196bb733fffff,89196bb7337ffff,8a196bb73347fff,88196bb733fffff/89196bb7337ffff/8a196bb73347ff...
1,1_00002,2024-06-25 05:40:31,88196bb733fffff,89196bb7337ffff,8a196bb73347fff,88196bb733fffff/89196bb7337ffff/8a196bb73347ff...
2,1_00003,2024-06-25 05:40:31,88196bb733fffff,89196bb7337ffff,8a196bb7334ffff,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...
3,1_00004,2024-06-25 05:40:31,88196bb733fffff,89196bb7337ffff,8a196bb7334ffff,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...
4,1_00005,2024-06-25 05:40:31,88196bb733fffff,89196bb7337ffff,8a196bb7334ffff,88196bb733fffff/89196bb7337ffff/8a196bb7334fff...


---
## 2. `internal/import_locations.txt` — panorama location metadata

Tab-delimited. Contains `recording_*` filenames with XYZ coordinates and
orientation angles. Note: Dutch locale — decimal separator is a **comma**, not a dot.

In [7]:
# Read first 5 rows; sep='\t' for tab-delimited, decimal=',' for Dutch locale
loc_peek = pd.read_csv(IMPORT_LOC_PATH, sep="\t", nrows=5, decimal=",")

print("=== Columns and dtypes ===")
print(loc_peek.dtypes)
print()
print("=== First 5 rows ===")
loc_peek

=== Columns and dtypes ===
Filename         str
X            float64
Y            float64
Z            float64
Pan          float64
Tilt         float64
Roll         float64
Timestamp      int64
dtype: object

=== First 5 rows ===


,Filename,X,Y,Z,Pan,Tilt,Roll,Timestamp
0,recording_2024-06-25_05-40-31_00001,98448.887,438163.582,0.068,0.0,0.0,0.0,1719294029
1,recording_2024-06-25_05-40-31_00002,98444.797,438160.764,-0.028,0.0,0.0,0.0,1719294036
2,recording_2024-06-25_05-40-31_00003,98441.052,438157.438,0.258,0.0,0.0,0.0,1719294041
3,recording_2024-06-25_05-40-31_00004,98440.239,438152.507,0.574,0.0,0.0,0.0,1719294043
4,recording_2024-06-25_05-40-31_00005,98441.301,438147.582,0.708,0.0,0.0,0.0,1719294044


In [8]:
# Row count via line counting (same approach as above)
with open(IMPORT_LOC_PATH, "r", encoding="utf-8") as f:
    n_rows_loc = sum(1 for _ in f) - 1

print(f"Total rows in import_locations.txt: {n_rows_loc:,}")

Total rows in import_locations.txt: 618,600


In [9]:
# Check the Filename format — expect 'recording_YYYY-MM-DD_HH-MM-SS_NNNNN'
# and verify that Pan is uniformly 0 (north-facing panoramas)
print("Filename sample:", loc_peek["Filename"].iloc[0])
print("Pan unique values:", loc_peek["Pan"].unique())

Filename sample: recording_2024-06-25_05-40-31_00001
Pan unique values: [0.]


---
## 3. `internal/Panorama_2024_RD/Panorama_2024_RD.shp` — panorama shapefile

Point shapefile in RD New (EPSG:28992). Contains the same `recording_*` filenames
as `import_locations.txt`, plus X/Y coordinates. Already used in notebook 02.

In [10]:
# Use fiona to read the schema and feature count without loading the full geometry.
# This gives us column names, types, and CRS for free.
with fiona.open(SHAPEFILE_PATH) as src:
    print("CRS:    ", src.crs)
    print("Count:  ", len(src))
    print("Schema: ", src.schema)
    print("Driver: ", src.driver)

CRS:     EPSG:28992
Count:   618600
Schema:  {'properties': {'Filename': 'str:50', 'X': 'float:31.15', 'Y': 'float:31.15'}, 'geometry': 'Point'}
Driver:  ESRI Shapefile


In [11]:
# Read just 5 features as a GeoDataFrame using the rows= parameter.
# This avoids pulling in all 618k geometries.
shp_peek = gpd.read_file(SHAPEFILE_PATH, rows=5)

print("=== Columns and dtypes ===")
print(shp_peek.dtypes)
print()
print("=== First 5 rows ===")
shp_peek

=== Columns and dtypes ===
Filename         str
X            float64
Y            float64
geometry    geometry
dtype: object

=== First 5 rows ===


,Filename,X,Y,geometry
0,recording_2024-06-25_05-40-31_00001,98448.887,438163.582,POINT (9.84e+04 4.38e+05)
1,recording_2024-06-25_05-40-31_00002,98444.797,438160.764,POINT (9.84e+04 4.38e+05)
2,recording_2024-06-25_05-40-31_00003,98441.052,438157.438,POINT (9.84e+04 4.38e+05)
3,recording_2024-06-25_05-40-31_00004,98440.239,438152.507,POINT (9.84e+04 4.38e+05)
4,recording_2024-06-25_05-40-31_00005,98441.301,438147.582,POINT (9.84e+04 4.38e+05)


In [12]:
# Verify Filename format matches import_locations.txt
print("Shapefile Filename sample: ", shp_peek["Filename"].iloc[0])

Shapefile Filename sample:  recording_2024-06-25_05-40-31_00001


---
## 5. Summary

| File | Rows | Key column(s) | Role |
|---|---|---|---|
| `image_index.csv` | ~617k | `image_id`, `trip_time`, `H3_8/9/10` | Maps any location → H3 folder path → server file |
| `import_locations.txt` | ~618k | `Filename` (recording_*) | XYZ + orientation per panorama; links via decoded trip_time + seq |
| `Panorama_2024_RD.shp` | ~618k | `Filename` (recording_*) | Geometry-aware version of the above; same Filename key |

**For the export task:** load `image_index.csv` once, build a `{recording_key: server_path}` lookup dict, then batch-resolve the ~713 selected photos from notebook 04/05 output.